# IBM Employee Attrition Analysis — Classification Modeling

**Dataset:** IBM HR Analytics Employee Attrition (1,470 employees, 35 variables)  
**Tools:** Python · scikit-learn · pandas · seaborn · SMOTE / ADASYN

---

## Notebook Overview

This is the main modeling notebook in the IBM attrition project. Using the prepared data from `01_Data_Preparation_and_EDA.ipynb`, it compares classification approaches for distinguishing attrition from non-attrition cases.

**Models covered:**
- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- Comparison across original, SMOTE-balanced, and ADASYN-balanced datasets

**Key techniques:**
- Train/test split and model evaluation
- Handling class imbalance (attrition base rate ~16%) using oversampling
- Feature importance extraction from tree-based models
- Comparing interpretable and higher-capacity classifiers

**Project context:** The emphasis is on demonstrating a clear modeling workflow and careful interpretation rather than treating benchmark results as deployment guidance.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier 
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import * # import all libraries under sklearn.metrics
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.metrics import *
plt.rc("font", size=14)

sns.set(style="white")
sns.set(style="whitegrid", color_codes=True)



# See #1137: this allows compatibility for scikit-learn >= 0.24
import sklearn.utils
from sklearn.utils import safe_indexing
# except ImportError:
# from sklearn.utils import _safe_indexing

# Oversample and plot imbalanced dataset with ADASYN
from collections import Counter
from imblearn.over_sampling import ADASYN
from imblearn.over_sampling import SMOTE 
from imblearn.under_sampling import RandomUnderSampler
from matplotlib import pyplot
from numpy import where


%matplotlib inline


# 2 functions to print out metrics (written by us, not in sklearn)

# define a function for calculating the metric to be used later 
# takes in 2 inputs: Y_pred, Y_true
# and uses them to calculate metrics using functions in sklearn
def classification_metrics(Y_pred, Y_true):
    acc = accuracy_score(Y_true, Y_pred)
    precision = precision_score(Y_true, Y_pred)
    recall = recall_score(Y_true, Y_pred)
    f1score = f1_score(Y_true, Y_pred)
    auc = roc_auc_score(Y_true, y_pred)

    # the function's outputs are the 5 variables below
    return acc, precision, recall, f1score, auc

# define a function for printing the metrics using inputs: classifierName, Y_pred, Y_true
# e.g. inputs can be: 'Logistic Regression', y_pred, y_test
# inside the function, we do something with the inputs (e.g. run classification_metrics on the inputs)
# classification_metrics is antoher function we wrote above
def display_metrics(classifierName, Y_pred, Y_true):
    print ("______________________________________________")
    print ("Model: "+classifierName)
    acc, precision, recall, f1score, auc = classification_metrics(Y_pred, Y_true)
    # returns 5 vars: acc, precision, recall, f1score, auc
    # print them below
    print ("Accuracy: "+str(acc))
    print ("Precision: "+str(precision))
    print ("Recall: "+str(recall))
    print ("F1-score: "+str(f1score))
    print ("AUC: "+str(auc))
    print ("______________________________________________")
    print ("")






In [ ]:
#import data 
df = pd.read_csv('HR_Attrition_IBM.csv')
df

df0 = df
df.info()

# Pre - Processing

In [ ]:
df.columns
df.shape # Columns and rows 
#1470 Rows X 35 Columns (variables)

## Missing Data


In [ ]:
print(df.isnull()) # for each cell, print True/False (True = missing Data)

#df[df['player'].isnull()] # filtering that keeps rows with missing df.purpose for a variable

#df.dropna() # drop any row with ANY missing value for any feature in a row. 
df #NO MISSING DATA
print(len(df)) # NO MISSING DATA
print(df.shape)


# NO MISSING DATA



In [ ]:
#see number of rows of a dataframe or variable 
print(len(df)) # number of rows 
#print(len(df.***VARIABLE**)) # number of rows in a var (purpose)


# Needless Data

In [ ]:
# ---Removing Variable - "Over18" - all enteries are over 18--#
# df.Over18
df.Over18.unique()
df = df.drop('Over18', axis=1) # axis=1 indicates that 'new' is a column 

# ---Removing Variable - "StandardHours" - all enteries are 80---#
# df.StandardHours
df.StandardHours.unique()
df = df.drop('StandardHours', axis=1) # axis=1 indicates that 'new' is a column 

# ---Removing Variable - "EmployeeCount" - all enteries are counts ---#
df = df.drop('EmployeeCount', axis=1) # axis=1 indicates that 'new' is a column 
# ---Removing Variable - "EmployeeNumber" - all enteries are 1---#
df = df.drop('EmployeeNumber', axis=1) # axis=1 indicates that 'new' is a column 


# [1470 rows x 31 columns - 30 IV + 1 DV]

In [ ]:
df.info()


In [ ]:
# Converting Independent Variable DataTypes 


##Boolean/Categorical Data(attrition, Gender, Overtime)

##---OverTime----##
# df['OverTime'] = df.OverTime.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'
df.loc[df["OverTime"] == "Yes", "OverTime"] = 1
df.loc[df["OverTime"] == "No", "OverTime"] = 0
##---Gender----##
# df['Gender'] = df.Gender.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'
df.loc[df["Gender"] == "Male", "Gender"] = 1
df.loc[df["Gender"] == "Female", "Gender"] = 0


##Nominal Data/Ordinal Data/ Interval Data/ Ratio Data (PerformanceRating, etc.))

# print(df.info())
print(df)



#[1470 rows x 33 columns]



### Assumption #1: The Response/Dependent Variable is Binary -- ## Data Type

In [ ]:
# Converting Dependent Variable DataType

##Boolean/Categorical Data(attrition, Gender, Overtime)

##---Attrition----##
# df['Attrition'] = df.Attrition.astype('bool') #convert var 'date'to a date data type in our dataframe 'df'

# df.loc[df["Attrition"] == "Yes", "Attrition"] = 1
# df.loc[df["Attrition"] == "No", "Attrition"] = 0
df["Attrition"] = np.where(df["Attrition"] == "No", 0, 1)

##Nominal Data/Ordinal Data/ Interval Data/ Ratio Data (PerformanceRating, etc.))

# print(df.info())
# print(df)

df
#[1470 rows x 33 columns]


In [ ]:
df.Attrition.value_counts()



In [ ]:
count_no_At = len(df[df['Attrition']==0])
count_At = len(df[df['Attrition']==1])
pct_of_no_At = count_no_At/(count_no_At+count_At)
print("percentage of no Attrition is", pct_of_no_At*100)
pct_of_At = count_At/(count_no_At+count_At)
print("percentage of Attrition", pct_of_At*100)

In [ ]:
#check False/ No Attrition
1233/(1233+237)

# df.Attrition



In [ ]:
sns.countplot(x='Attrition', data=df, palette='hls')
plt.show()
# plt. savefig('count plot')

In [ ]:
total = float(len(df))
ax = sns.countplot(x='Attrition', data=df, palette='hls')
plt.title('Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()



# Decision Tree

## Training a Decision Tree Model
- Use DecisionTreeClassifier in sklearn 
- Some paramters: 
    - criterion: The function to measure the quality of a split. Supported criteria are “gini” for the Gini impurity and “entropy” for the information gain.
    - max_depth: The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
    - min_samples_leaf: The minimum number of samples required to be at a leaf node. 
    - max_features: The number of features to consider when looking for the best split:
    - see all parameters in the documentation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

### 1. Balancing the DV (Attrition) with our IV's

In [ ]:
# Replicate original data so not to overwrite it 
df1 = df.copy()
dfs = df.copy()


df1 #[1470 rows × 31 columns]

In [ ]:
total = float(len(df1))
ax = sns.countplot(x='Attrition', data=df1, palette='hls')
plt.title('Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()

In [ ]:
df1

In [ ]:
# create dummy variables for each country using function pd.get_dummies() for variable 'Country'
# add a prefix to names of dummies using 'prefix='Country'
# embed pd.get_dummies(ad_data2['Country'], prefix='Country') inside pd.conat()

# another way:
# countries = pd.get_dummies(ad_data2['Country'], prefix='Country')
# ad_data2 = pd.concat([ad_data2, countries],axis=1)


##-----BusinessTravel------###
df1 = pd.concat([df1, pd.get_dummies(df1['BusinessTravel'], prefix='BusinessTravel')],axis=1)
##-----Department------###
df1 = pd.concat([df1, pd.get_dummies(df1['Department'], prefix='Department')],axis=1)
##-----EducationField------###
df1 = pd.concat([df1, pd.get_dummies(df1['EducationField'], prefix='EducationField')],axis=1)
##-----JobRole------###
df1 = pd.concat([df1, pd.get_dummies(df1['JobRole'], prefix='JobRole')],axis=1)
##-----MaritalStatus------###
df1 = pd.concat([df1, pd.get_dummies(df1['MaritalStatus'], prefix='MaritalStatus')],axis=1)

In [ ]:
### now drop the original 'country' column (you don't need it anymore)


##-----BusinessTravel------###
df1.drop(['BusinessTravel'],axis=1, inplace=True)
##-----Department------###
df1.drop(['Department'],axis=1, inplace=True)

##-----EducationField------###
df1.drop(['EducationField'],axis=1, inplace=True)

##-----JobRole------###
df1.drop(['JobRole'],axis=1, inplace=True)

##-----MaritalStatus------###
df1.drop(['MaritalStatus'],axis=1, inplace=True)


In [ ]:
df1.info()
df1

# 1470 rows × 50 columns w/ Attrition(DV)



In [ ]:
df1.columns
# 1470 rows × 50 columns



In [ ]:
df1

In [ ]:
# Create a list of predictor (x) variables: just Age
predictors1 = ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager', 'BusinessTravel_Non-Travel',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Human Resources', 'Department_Research & Development',
       'Department_Sales', 'EducationField_Human Resources',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Research Scientist',
       'JobRole_Sales Executive', 'JobRole_Sales Representative',
       'MaritalStatus_Divorced', 'MaritalStatus_Married',
       'MaritalStatus_Single']


# Create another list of predictor variables: Age, Country dummies (without the first country dummy) 
# [i for i in ad_data2.columns if i.startswith('Country')]: chooses all items in ad_data2.columns (var names)
# which start with 'Country'
# [i for i in ad_data2.columns if i.startswith('Country')][1:] -> add all countries but the first one (drop country at index 0)


###########predictors2 = ['Age']+[i for i in df1.columns if i.startswith('BusinessTravel')][1:]+[i for i in df1.columns if i.startswith('Department')][1:]+[i for i in df1.columns if i.startswith('EducationField')][1:]+[i for i in df1.columns if i.startswith('JobRole')][1:]+[i for i in df1.columns if i.startswith('MaritalStatus')][1:]


##-----BusinessTravel------###
##-----Department------###
##-----EducationField------###
##-----JobRole------###
##-----MaritalStatus------###




In [ ]:
# create dataframes for X (using Age only) and y variables 
X = df1[predictors1] # choose predictors1
y = df1['Attrition'] # choose target var

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


### Unbalanced DV: Over-Sampling with SMOTE

SMOTE: Synthetic Minority Over sampling Technique (SMOTE) algorithm applies KNN approach where it selects K nearest neighbors, joins them and creates the synthetic samples in the space. The algorithm takes the feature vectors and its nearest neighbors, computes the distance between these vectors. The difference is multiplied by random number between (0, 1) and it is added back to feature. SMOTE algorithm is a pioneer algorithm and many other algorithms are derived from SMOTE.


In [ ]:
# extract new x-vars after dimensionality reduction
X = df1[predictors1]
y = df1['Attrition']

# X = [1470 rows × 49 columns/IVs]
# y = [1470 rows × 1 column/DV]

In [ ]:
# X
# y

# X = [1470 rows × 51 columns/IVs]
# y = [1470 rows × 1 column/DV]

In [ ]:
sm = SMOTE(random_state=101)
X_res, y_res = sm.fit_resample(X, y)

In [ ]:
# split data into train/test data
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state = 101)

model = DecisionTreeClassifier(criterion='entropy', random_state=101, max_depth=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

confusion_matrix_results = confusion_matrix(y_pred, y_test)

display_metrics('SMOTE - Decision Tree Classifier', y_pred, y_test)

In [ ]:
SMOTElabels = pd.DataFrame(y_res) # label is the value of the target var
SMOTEfeatures = pd.DataFrame (X_res)

In [ ]:
SMOTEdata = pd.concat([SMOTElabels, SMOTEfeatures], axis=1)
SMOTEdata

print(isinstance(SMOTEdata, pd.DataFrame))

SMOTEdata.to_csv('HR_Attrition_IBM.csv')


# [2466 rows × 52 columns]



In [ ]:
total = float(len(SMOTEdata))
ax = sns.countplot(x='Attrition', data=SMOTEdata, palette='hls')
plt.title('SMOTE - Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()

### Unbalanced DV: Over-Sampling with ADASYN

ADAptive SYNthetic (ADASYN) is based on the idea of adaptively generating minority data samples according to their distributions using K nearest neighbor. The algorithm adaptively updates the distribution and there are no assumptions made for the underlying distribution of the data.  The algorithm uses Euclidean distance for KNN Algorithm. The key difference between ADASYN and SMOTE is that the former uses a density distribution, as a criterion to automatically decide the number of synthetic samples that must be generated for each minority sample by adaptively changing the weights of the different minority samples to compensate for the skewed distributions. The latter generates the same number of synthetic samples for each original minority sample.

In [ ]:
# extract new x-vars after dimensionality reduction
X = df1[predictors1]
y = df1['Attrition']

# X = [1470 rows × 51 columns/IVs]
# y = [1470 rows × 1 column/DV]

In [ ]:
ada = ADASYN(random_state=101)
X_res, y_res = ada.fit_resample(X, y)


In [ ]:
# split data into train/test data
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state = 101)

model = DecisionTreeClassifier(criterion='entropy', random_state=101, max_depth=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

confusion_matrix_results = confusion_matrix(y_pred, y_test)

display_metrics('ADASYN - Decision Tree Classifier', y_pred, y_test)

In [ ]:
ADAlabels = pd.DataFrame(y_res) # label is the value of the target var
ADAfeatures = pd.DataFrame (X_res)

In [ ]:
ADAdata = pd.concat([ADAlabels, ADAfeatures], axis=1)
ADAdata

# [2406 rows × 52 columns]



In [ ]:
ADAdata = pd.concat([ADAlabels, ADAfeatures], axis=1)
ADAdata

print(isinstance(ADAdata, pd.DataFrame))

ADAdata.to_csv('HR_Attrition_IBM.csv')


# [2406 rows × 52 columns]



In [ ]:
total = float(len(ADAdata))
ax = sns.countplot(x='Attrition', data=ADAdata, palette='hls')
plt.title('ADASYN - Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()

# Standarizing ALL non-Categorical Predictors

### SMOTE Data

In [ ]:
SMOTEdata

In [ ]:
df = SMOTEdata.copy()
df

In [ ]:
print(isinstance(df, pd.DataFrame))
df


In [ ]:
df.columns

In [ ]:
# # # All 49 Predictors: ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
#        'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
#        'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
#        'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike',
#        'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
#        'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
#        'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
#        'YearsWithCurrManager', 'BusinessTravel_Non-Travel',
#        'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
#        'Department_Human Resources', 'Department_Research & Development',
#        'Department_Sales', 'EducationField_Human Resources',
#        'EducationField_Life Sciences', 'EducationField_Marketing',
#        'EducationField_Medical', 'EducationField_Other',
#        'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
#        'JobRole_Human Resources', 'JobRole_Laboratory Technician',
#        'JobRole_Manager', 'JobRole_Manufacturing Director',
#        'JobRole_Research Director', 'JobRole_Research Scientist',
#        'JobRole_Sales Executive', 'JobRole_Sales Representative',
#        'MaritalStatus_Divorced', 'MaritalStatus_Married',
#        'MaritalStatus_Single']



In [ ]:
# Categorical Predictors -- NO standardization Needed
#-----OverTime------###
#-----Gender------#### 
##-----BusinessTravel------###
##-----Department------###
##-----EducationField------###
##-----JobRole------###
##-----MaritalStatus------###

In [ ]:
# Continuous Predictors -- Standardization Needed
#-----Age------###
#-----DailyRate------###
#-----DistanceFromHome------###
#-----Education------###
#-----EnvironmentSatisfaction------###
#-----HourlyRate------###
#-----JobInvolvement------###
#-----JobLevel------###
#-----JobSatisfaction------###
#-----MonthlyIncome------###
#-----MonthlyRate------###
#-----NumCompaniesWorked------###
#-----PercentSalaryHike------###
#-----PerformanceRating------###
#-----RelationshipSatisfaction------###
#-----StockOptionLevel------###
#-----TotalWorkingYears------###
#-----TrainingTimesLastYear------###
#-----WorkLifeBalance------###
#-----YearsAtCompany------###
#-----YearsInCurrentRole------###
#-----YearsSinceLastPromotion------###
#-----YearsWithCurrManager------###




In [ ]:
df.columns

In [ ]:
CF = ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']

BF = ['Attrition','Gender', 'OverTime', 'BusinessTravel_Non-Travel',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Human Resources', 'Department_Research & Development',
       'Department_Sales', 'EducationField_Human Resources',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Research Scientist',
       'JobRole_Sales Executive', 'JobRole_Sales Representative',
       'MaritalStatus_Divorced', 'MaritalStatus_Married',
       'MaritalStatus_Single']


In [ ]:
features = df[CF] # choose predictors to standardize
y = df[BF] # choose rest features+DV

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:
features = preprocessing.scale(features)
features

In [ ]:
features = pd.DataFrame (features)

In [ ]:
features.info()

In [ ]:
features.columns =['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']

# 23 columns


In [ ]:
features.info()

In [ ]:
labels = pd.DataFrame(y) # convert non standardized features to DF

In [ ]:
data = pd.concat([labels, features], axis=1)
data

In [ ]:
data.info()

In [ ]:
data.to_csv('St_SMOTE_HR_Data.csv')


### ADASYN Data

In [ ]:
ADAdata.info()

In [ ]:
df = ADAdata.copy()
df

In [ ]:
print(isinstance(df, pd.DataFrame))
df


In [ ]:
df.columns

In [ ]:
# # # All 49 Predictors: ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
#        'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
#        'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
#        'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike',
#        'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
#        'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
#        'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
#        'YearsWithCurrManager', 'BusinessTravel_Non-Travel',
#        'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
#        'Department_Human Resources', 'Department_Research & Development',
#        'Department_Sales', 'EducationField_Human Resources',
#        'EducationField_Life Sciences', 'EducationField_Marketing',
#        'EducationField_Medical', 'EducationField_Other',
#        'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
#        'JobRole_Human Resources', 'JobRole_Laboratory Technician',
#        'JobRole_Manager', 'JobRole_Manufacturing Director',
#        'JobRole_Research Director', 'JobRole_Research Scientist',
#        'JobRole_Sales Executive', 'JobRole_Sales Representative',
#        'MaritalStatus_Divorced', 'MaritalStatus_Married',
#        'MaritalStatus_Single']



In [ ]:
# Categorical Predictors -- NO standardization Needed
#-----OverTime------###
#-----Gender------#### 
##-----BusinessTravel------###
##-----Department------###
##-----EducationField------###
##-----JobRole------###
##-----MaritalStatus------###

In [ ]:
# Continuous Predictors -- Standardization Needed
#-----Age------###
#-----DailyRate------###
#-----DistanceFromHome------###
#-----Education------###
#-----EnvironmentSatisfaction------###
#-----HourlyRate------###
#-----JobInvolvement------###
#-----JobLevel------###
#-----JobSatisfaction------###
#-----MonthlyIncome------###
#-----MonthlyRate------###
#-----NumCompaniesWorked------###
#-----PercentSalaryHike------###
#-----PerformanceRating------###
#-----RelationshipSatisfaction------###
#-----StockOptionLevel------###
#-----TotalWorkingYears------###
#-----TrainingTimesLastYear------###
#-----WorkLifeBalance------###
#-----YearsAtCompany------###
#-----YearsInCurrentRole------###
#-----YearsSinceLastPromotion------###
#-----YearsWithCurrManager------###




In [ ]:
df.columns

In [ ]:
CF = ['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']

BF = ['Attrition','Gender', 'OverTime', 'BusinessTravel_Non-Travel',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Human Resources', 'Department_Research & Development',
       'Department_Sales', 'EducationField_Human Resources',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Research Scientist',
       'JobRole_Sales Executive', 'JobRole_Sales Representative',
       'MaritalStatus_Divorced', 'MaritalStatus_Married',
       'MaritalStatus_Single']


In [ ]:
features = df[CF] # choose predictors to standardize
y = df[BF] # choose rest features+DV

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:
features = preprocessing.scale(features)
features

In [ ]:
features = pd.DataFrame (features)

In [ ]:
features.info()

In [ ]:
features.columns =['Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']

# 23 columns


In [ ]:
features.info()

In [ ]:
labels = pd.DataFrame(y) # convert non standardized features to DF

In [ ]:
data = pd.concat([labels, features], axis=1)
data

In [ ]:
data.info()

In [ ]:
data.to_csv('St_ADASYN_HR_Data.csv')


# Decision Tree Model - Standardized SMOTE Data

In [ ]:
#import data 
df = pd.read_csv('St_SMOTE_HR_Data.csv')
df.info()
df.columns

In [ ]:
# ---Removing Variable - "Unnamed: 0" - Saved Index--#
df = df.drop('Unnamed: 0', axis=1) # axis=1 indicates that 'new' is a column 
df.info()

In [ ]:
df.Attrition.value_counts()
# balanced DV check

In [ ]:
df

In [ ]:
count_no_At = len(df[df['Attrition']==0])
count_At = len(df[df['Attrition']==1])
pct_of_no_At = count_no_At/(count_no_At+count_At)
print("percentage of no Attrition is", pct_of_no_At*100)
pct_of_At = count_At/(count_no_At+count_At)
print("percentage of Attrition", pct_of_At*100)

In [ ]:
total = float(len(df))
ax = sns.countplot(x='Attrition', data=df, palette='hls')
plt.title('Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()



In [ ]:
# Specify Predictors 

predictors1 = ['Gender', 'OverTime', 'BusinessTravel_Non-Travel',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Human Resources', 'Department_Research & Development',
       'Department_Sales', 'EducationField_Human Resources',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Research Scientist',
       'JobRole_Sales Executive', 'JobRole_Sales Representative',
       'MaritalStatus_Divorced', 'MaritalStatus_Married',
       'MaritalStatus_Single','Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']





In [ ]:
# creating dataframes for X and y variables 

X = df[predictors1] # choose predictors1
y = df['Attrition'] # choose target var

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:
# Split data - test_size of 30%, random_state=101
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101, stratify=y)


In [ ]:
X_train.head()
y_train.head()

In [ ]:
# train model, set criterion, max_depth=5 (no tree over 5 levels)
model_dt = DecisionTreeClassifier(criterion='entropy', random_state=101)

# fit the model
model_dt.fit(X_train, y_train)


y_pred = model_dt.predict(X_test)

confusion_matrix_results = confusion_matrix(y_pred, y_test)

print('confusion matrix: \n', confusion_matrix_results)

display_metrics('S_SMOTE - Decision Tree Classifier', y_pred, y_test)


In [ ]:
# train model, set criterion, max_depth=5 (no tree over 5 levels)
model_dt5 = DecisionTreeClassifier(criterion='entropy', random_state=101, max_depth=5)

# fit the model
model_dt5.fit(X_train, y_train)


y_pred = model_dt5.predict(X_test)

confusion_matrix_results = confusion_matrix(y_pred, y_test)
print('confusion matrix: \n', confusion_matrix_results)

display_metrics('S_SMOTE - Decision Tree Classifier', y_pred, y_test)



## Training a Decision Tree Model
- Use DecisionTreeClassifier in sklearn 
- Some paramters: 
    - criterion: The function to measure the quality of a split. Supported criteria are “gini” for the Gini impurity and “entropy” for the information gain.
    - max_depth: The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
    - min_samples_leaf: The minimum number of samples required to be at a leaf node. 
    - max_features: The number of features to consider when looking for the best split:
    - see all parameters in the documentation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

## Feature Importance

In [ ]:
# visualize x-vars by importance
importances = model_dt.feature_importances_  # extract importance metrics 
indices = np.argsort(importances)[::-1] # sorts the rows 

print("Feature ranking:")

feature_names = X_train.columns 

# create a dataframe using Pandas for the vars using pd.DataFrame() using feature names and importance metrics
fi = pd.DataFrame([feature_names[indices[0:10]], importances[indices][0:10]])
fi = fi.T
fi.columns = ['Feature', 'Total Reduction of Criterion']

print(fi)

# Plot the feature importances of the forest using seaborn
plt.figure()
plt.title("Feature importances")

sns.set_color_codes("pastel")
sns.barplot(x="Total Reduction of Criterion", y="Feature", data=fi, color="b")


## Predictions and Evaluation of Decision Tree

In [ ]:
# use predict function to make predictions 
y_pred = model_dt.predict(X_test)

In [ ]:
# calculate the confusion matrix for the test data 
confusion_matrix_results = confusion_matrix(y_test, y_pred)

# print the counts of the confusion matrix 
print('confusion matrix: \n', confusion_matrix_results)

# print the metrics 
display_metrics('Decision Tree', y_pred, y_test)


In [ ]:
# predict each instance's probability of survival for each individual using 'predict_proba' function
model_dt.predict_proba(X_train)


# Decision Tree Model - Standardized ADASYN Data

In [ ]:
#import data 
df = pd.read_csv('St_ADASYN_HR_Data.csv')
df.info()
df.columns

In [ ]:
# ---Removing Variable - "Unnamed: 0" - Saved Index--#
df = df.drop('Unnamed: 0', axis=1) # axis=1 indicates that 'new' is a column 
df.info()

In [ ]:
df.Attrition.value_counts()
# balanced DV check

In [ ]:
df

In [ ]:
count_no_At = len(df[df['Attrition']==0])
count_At = len(df[df['Attrition']==1])
pct_of_no_At = count_no_At/(count_no_At+count_At)
print("percentage of no Attrition is", pct_of_no_At*100)
pct_of_At = count_At/(count_no_At+count_At)
print("percentage of Attrition", pct_of_At*100)

In [ ]:
total = float(len(df))
ax = sns.countplot(x='Attrition', data=df, palette='hls')
plt.title('Attrition Class Distribution', fontsize=20)
for p in ax.patches:
    ax.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01), fontsize=13)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width()
    y = p.get_height()
    ax.annotate(percentage, (x, y),ha='center', fontsize=13)
plt.show()



In [ ]:
# Specify Predictors 

predictors1 = ['Gender', 'OverTime', 'BusinessTravel_Non-Travel',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Human Resources', 'Department_Research & Development',
       'Department_Sales', 'EducationField_Human Resources',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'JobRole_Healthcare Representative',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Research Scientist',
       'JobRole_Sales Executive', 'JobRole_Sales Representative',
       'MaritalStatus_Divorced', 'MaritalStatus_Married',
       'MaritalStatus_Single','Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate',
       'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']





In [ ]:
# creating dataframes for X and y variables 

X = df[predictors1] # choose predictors1
y = df['Attrition'] # choose target var

# see list of X variables 
# X.columns is the list of var names
# [i for i in X.columns]: choose all the items in X.columns (var names in X) in list
print('X variables:\n', [i for i in X.columns])


In [ ]:
# Split data - test_size of 30%, random_state=101
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101, stratify=y)


In [ ]:
X_train.head()
y_train.head()

In [ ]:
# train model, set criterion, max_depth=5 (no tree over 5 levels)
model_dt = DecisionTreeClassifier(criterion='entropy', random_state=101)

# fit the model
model_dt.fit(X_train, y_train)


y_pred = model_dt.predict(X_test)

confusion_matrix_results = confusion_matrix(y_pred, y_test)

display_metrics('S_SMOTE - Decision Tree Classifier', y_pred, y_test)


In [ ]:
# train model, set criterion, max_depth=5 (no tree over 5 levels)
model_dt5 = DecisionTreeClassifier(criterion='entropy', random_state=101, max_depth=500000)

# fit the model
model_dt5.fit(X_train, y_train)


y_pred = model_dt5.predict(X_test)

confusion_matrix_results = confusion_matrix(y_pred, y_test)

display_metrics('S_SMOTE - Decision Tree Classifier', y_pred, y_test)



## Feature Importance

In [ ]:
# visualize x-vars by importance
importances = model_dt.feature_importances_  # extract importance metrics 
indices = np.argsort(importances)[::-1] # sorts the rows 

print("Feature ranking:")

feature_names = X_train.columns 

# create a dataframe using Pandas for the vars using pd.DataFrame() using feature names and importance metrics
fi = pd.DataFrame([feature_names[indices[0:10]], importances[indices][0:10]])
fi = fi.T
fi.columns = ['Feature', 'Total Reduction of Criterion']

print(fi)

# Plot the feature importances of the forest using seaborn
plt.figure()
plt.title("Feature importances")

sns.set_color_codes("pastel")
sns.barplot(x="Total Reduction of Criterion", y="Feature", data=fi, color="b")


## Predictions and Evaluation of Decision Tree

In [ ]:
# use predict function to make predictions 
y_pred = model_dt.predict(X_test)

In [ ]:
# calculate the confusion matrix for the test data 
confusion_matrix_results = confusion_matrix(y_test, y_pred)

# print the counts of the confusion matrix 
print('confusion matrix: \n', confusion_matrix_results)

# print the metrics 
display_metrics('Decision Tree', y_pred, y_test)


In [ ]:
# predict each instance's probability of survival for each individual using 'predict_proba' function
model_dt.predict_proba(X_train)


In [ ]:
clf = tree.DecisionTreeClassifier(random_state=101)
clf = clf.fit(df[predictors1],df['Attrition'])
tree.plot_tree(clf)

In [ ]:
import graphviz
df = tree.export_graphviz(clf, out_file=None, 
                                feature_names=predictors1,  
                                class_names='Attrition',
                                filled=True)

# Draw graph
graph = graphviz.Source(df, format="png") 
graph

